# 🔥 PyTorch ile Basit Veri Kümesi (Dataset) Oluşturma

[![Python](https://img.shields.io/badge/Python-3.8%2B-blue?logo=python)](https://www.python.org)
[![PyTorch](https://img.shields.io/badge/PyTorch-2.x-orange?logo=pytorch)](https://pytorch.org)
[![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)](https://opensource.org/licenses/MIT)

Bu notebook, PyTorch'un `Dataset` API'sini sıfırdan öğrenmek için hazırlanmıştır.  
Kendi veri kümenizi nasıl oluşturacağınızı, dönüşüm (transform) sınıfları yazacağınızı ve bunları `Compose` ile birleştireceğinizi adım adım göreceksiniz.

## 📋 İçindekiler

| # | Konu | Açıklama |
|---|------|----------|
| 1 | [Kurulum & İmportlar](#1-kurulum--i̇mportlar) | Gerekli kütüphanelerin yüklenmesi |
| 2 | [Basit Dataset Oluşturma](#2-basit-dataset-oluşturma) | `Dataset` sınıfından miras alma |
| 3 | [Transform Sınıfları](#3-transform-sınıfları) | Veri dönüşüm işlemleri |
| 4 | [Compose ile Zincirleme](#4-compose-ile-zincirleme) | Birden fazla transform birleştirme |
| 5 | [Alıştırmalar](#5-alıştırmalar) | Pratik sorular ve çözümler |

---

## 🎯 Öğrenim Hedefleri

Bu notebook'u tamamladığınızda şunları yapabileceksiniz:
- ✅ PyTorch `Dataset` sınıfını miras alarak özel veri kümesi oluşturma
- ✅ `__len__` ve `__getitem__` metodlarının amacını anlama
- ✅ Callable transform sınıfları yazma
- ✅ `torchvision.transforms.Compose` ile transform pipeline'ı kurma

⏱️ **Tahmini süre:** 30 dakika

---

## 1. Kurulum & İmportlar

In [ ]:
# Gerekli kütüphaneleri yüklüyoruz (ilk çalıştırmada birkaç dakika sürebilir)
# CPU sürümünü kullanıyoruz — GPU'nuz varsa pytorch.org'dan uygun sürümü seçin
%pip install torch torchvision --quiet

In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision import transforms

# Tekrar üretilebilirlik için rastgele sayı üretecini sabitleyelim
torch.manual_seed(1)

print(f"PyTorch sürümü : {torch.__version__}")
print(f"CUDA kullanılabilir: {torch.cuda.is_available()}")

---

## 2. Basit Dataset Oluşturma

PyTorch'ta bir veri kümesi oluşturmak için `torch.utils.data.Dataset` sınıfından miras alınır.  
Alt sınıfın **mutlaka** iki metod tanımlaması gerekir:

| Metod | Görev |
|-------|-------|
| `__len__(self)` | Veri kümesindeki örnek sayısını döndürür; `len(dataset)` çağrısını karşılar |
| `__getitem__(self, idx)` | `idx` indeksindeki örneği döndürür; `dataset[i]` sözdizimini karşılar |

In [ ]:
class toy_set(Dataset):
    """
    Öğrenme amaçlı basit bir oyuncak veri kümesi.

    Parametreler
    ----------
    length : int
        Veri kümesindeki örnek sayısı (varsayılan: 100)
    transform : callable, optional
        Her örneğe uygulanacak dönüşüm fonksiyonu/sınıfı

    Örnekler
    --------
    Her X değeri [2, 2] tensörü, her Y değeri [1] tensörüdür.
    """

    def __init__(self, length=100, transform=None):
        self.len = length
        self.x = 2 * torch.ones(length, 2)   # Girdi özellikleri: shape (N, 2)
        self.y = torch.ones(length, 1)        # Hedef etiketler:   shape (N, 1)
        self.transform = transform

    def __getitem__(self, index):
        sample = self.x[index], self.y[index]
        if self.transform:
            sample = self.transform(sample)
        return sample

    def __len__(self):
        return self.len

### 2.1 Veri Kümesini Keşfedelim

In [ ]:
our_dataset = toy_set()

print("Veri kümesi nesnesi :", our_dataset)
print("Toplam örnek sayısı :", len(our_dataset))
print("İndeks 0'daki örnek :", our_dataset[0])

In [ ]:
# İlk 3 örneği döngüyle inceleyelim
print("İlk 3 örnek:")
print("-" * 40)
for i in range(3):
    x, y = our_dataset[i]
    print(f"  [idx={i}]  x: {x.tolist()}  |  y: {y.tolist()}")

In [ ]:
# Dataset nesnesi doğrudan iterable'dır — DataLoader gibi davranır
print("Tüm örnekleri iterasyon ile yazdırıyoruz (ilk 5):")
for i, (x, y) in enumerate(our_dataset):
    if i >= 5:
        print("  ...")
        break
    print(f"  x: {x.tolist()}  y: {y.tolist()}")

---

## 3. Transform Sınıfları

**Transform**, bir örneği alıp dönüştürülmüş hâlde geri veren herhangi bir `callable` nesnedir.  
Genellikle `__call__` metoduna sahip bir sınıf olarak tanımlanır; bu sayede:
- Parametreler `__init__`'te saklanır
- Dönüşüm mantığı `__call__`'da uygulanır

### 3.1 `add_mult` — Toplama ve Çarpma Dönüşümü

Bu transform, girdiye (`x`) bir sabit ekler ve çıktıyı (`y`) başka bir sabit ile çarpar.

In [ ]:
class add_mult:
    """
    x'e `addx` ekler, y'yi `muly` ile çarpar.

    Parametreler
    ----------
    addx : float  → x'e eklenecek değer  (varsayılan: 1)
    muly : float  → y ile çarpılacak değer (varsayılan: 2)
    """

    def __init__(self, addx=1, muly=2):
        self.addx = addx
        self.muly = muly

    def __call__(self, sample):
        x, y = sample
        return x + self.addx, y * self.muly

In [ ]:
# Transform nesnesini ve ham veri kümesini oluşturalım
a_m = add_mult()
data_set = toy_set()

print("Dönüşüm öncesi vs sonrası karşılaştırması (ilk 5 örnek):")
print(f"{'İndeks':<8} {'Orijinal x':<18} {'Orijinal y':<12} {'Dönüştürülmüş x':<20} {'Dönüştürülmüş y'}")
print("-" * 78)

for i in range(5):
    x, y   = data_set[i]
    x_, y_ = a_m(data_set[i])
    print(f"{i:<8} {str(x.tolist()):<18} {str(y.tolist()):<12} {str(x_.tolist()):<20} {str(y_.tolist())}")

### 3.2 Transform'u Dataset'e Entegre Etmek

Transform'u her seferinde elle uygulamak yerine, `toy_set` constructor'una geçirebiliriz.  
Bu sayede `dataset[i]` çağrıldığında dönüşüm **otomatik** uygulanır.

In [ ]:
# Transform, Dataset'e yapısal olarak bağlandı
cust_data_set = toy_set(transform=a_m)

print("Ham veri kümesi vs. transform'lu veri kümesi:")
print("-" * 60)
for i in range(3):
    x,  y  = data_set[i]
    x_, y_ = cust_data_set[i]
    print(f"[{i}] Ham     → x: {x.tolist()}, y: {y.tolist()}")
    print(f"[{i}] Dönüşüm → x: {x_.tolist()}, y: {y_.tolist()}")
    print()

---

## 4. Compose ile Zincirleme

`torchvision.transforms.Compose`, birden fazla transform'u sırayla uygulayan bir wrapper'dır.  
İlk transform'un çıktısı bir sonrakinin girdisi olur:

```
ham_veri  →  [add_mult]  →  ara_veri  →  [mult]  →  son_veri
```

### 4.1 `mult` — Çarpma Dönüşümü

In [ ]:
class mult:
    """
    x ve y'yi `factor` ile çarpar.

    Parametreler
    ----------
    factor : float  → çarpan (varsayılan: 100)
    """

    def __init__(self, factor=100):
        self.factor = factor

    def __call__(self, sample):
        x, y = sample
        return x * self.factor, y * self.factor

### 4.2 Pipeline Oluşturma

In [ ]:
# İki transform'u zincirleyelim: önce add_mult, sonra mult
data_transform = transforms.Compose([
    add_mult(),   # x += 1, y *= 2
    mult(),       # x *= 100, y *= 100
])

print("Compose pipeline:", data_transform)

In [ ]:
# Üç farklı veri kümesini yan yana karşılaştıralım
compose_data_set = toy_set(transform=data_transform)

print("Üç veri kümesinin karşılaştırması (ilk 3 örnek):")
print("=" * 75)

for i in range(3):
    x,    y    = data_set[i]
    x_,   y_   = cust_data_set[i]
    x_co, y_co = compose_data_set[i]

    print(f"\n📌 İndeks {i}")
    print(f"  Ham         → x: {x.tolist():<15}  y: {y.tolist()}")
    print(f"  add_mult    → x: {x_.tolist():<15}  y: {y_.tolist()}")
    print(f"  add_mult+mult → x: {x_co.tolist():<13}  y: {y_co.tolist()}")

### 4.3 Hesaplama Doğrulaması

İndeks 0'daki `x = [2, 2]`, `y = [1]` için adım adım:

| Adım | İşlem | x | y |
|------|-------|---|---|
| Başlangıç | — | `[2, 2]` | `[1]` |
| `add_mult` | x += 1, y *= 2 | `[3, 3]` | `[2]` |
| `mult` | x *= 100, y *= 100 | `[300, 300]` | `[200]` |

---

## 5. Alıştırmalar

> 💡 Her alıştırmayı önce kendi başınıza çözmeyi deneyin, sonra çözüm hücresini açın.

### Alıştırma 1
50 uzunluğunda bir `toy_set` nesnesi oluşturun ve uzunluğunu yazdırın.

In [ ]:
# Kodunuzu buraya yazın


In [ ]:
# ✅ Çözüm
my_dataset = toy_set(length=50)
print("Veri kümesi uzunluğu:", len(my_dataset))   # Beklenen çıktı: 50

### Alıştırma 2
Her iki değere de önce `add=2` ekleyip ardından `mul=10` ile çarpan bir `my_add_mult` transform sınıfı yazın.  
Bu transform'u kullanan bir `toy_set` oluşturun ve ilk 3 örneği yazdırın.

In [ ]:
# Kodunuzu buraya yazın


In [ ]:
# ✅ Çözüm
class my_add_mult:
    def __init__(self, add=2, mul=10):
        self.add = add
        self.mul = mul

    def __call__(self, sample):
        x, y = sample
        return (x + self.add) * self.mul, (y + self.add) * self.mul

my_dataset = toy_set(transform=my_add_mult())
for i in range(3):
    x_, y_ = my_dataset[i]
    print(f"[{i}]  x: {x_.tolist()},  y: {y_.tolist()}")
# Beklenen: x = (2+2)*10 = [40, 40],  y = (1+2)*10 = [30]

### Alıştırma 3
`mult()` ve `add_mult()` fonksiyonlarını birleştirerek **önce** `mult()`, **sonra** `add_mult()` çalışacak bir pipeline oluşturun.  
Dönüştürülmüş veri kümesindeki ilk 3 örneği yazdırın.

In [ ]:
# Kodunuzu buraya yazın


In [ ]:
# ✅ Çözüm
my_compose = transforms.Compose([mult(), add_mult()])
my_transformed_dataset = toy_set(transform=my_compose)

for i in range(3):
    x_, y_ = my_transformed_dataset[i]
    print(f"[{i}]  x: {x_.tolist()},  y: {y_.tolist()}")
# Beklenen: x = 2*100+1 = [201,201],  y = 1*100*2 = [200]

---

## 📝 Özet

Bu notebook'ta öğrendiklerimiz:

1. **`Dataset` sınıfı** — `__len__` ve `__getitem__` ile özel veri kümesi tanımlama
2. **Transform nesneleri** — `__call__` ile callable sınıflar oluşturma
3. **`Compose`** — Birden fazla transform'u sıralı pipeline olarak zincirleme
4. **Entegrasyon** — Transform'ların Dataset constructor'ına geçirilmesi

### Sonraki Adımlar

- 📦 [`DataLoader`](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) — Dataset'i batch'lere bölme ve shuffle etme
- 🖼️ [`torchvision.datasets`](https://pytorch.org/vision/stable/datasets.html) — Hazır veri kümeleri (MNIST, CIFAR-10 vb.)
- ✂️ [`torchvision.transforms`](https://pytorch.org/vision/stable/transforms.html) — Görüntü için hazır transform'lar

---

*PyTorch Derin Öğrenme — Modül 1, Lab 3.1*